## Ingest Fact Data into Bronze Layer

In [0]:
# Import Required Libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [0]:
%run /Workspace/Users/yklynk@gmail.com/Azure_databricks_data_engineering_project_shopvista_ecomm/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema,gold_schema)

## ORDER ITEMS

In [0]:
# Widget
dbutils.widgets.text('catalog', 'shopvista', 'catalog')
dbutils.widgets.text('storage_account_name', 'storageshopvista', 'storage_account_name')
dbutils.widgets.text('container_name', 'shopvista-raw-data', 'container_namee')


In [0]:
# Retrieve widget values for catalog and storage account details
catalog = dbutils.widgets.get('catalog')
storage_account_name = dbutils.widgets.get('storage_account_name')
container_name = dbutils.widgets.get('container_name')
print(catalog, storage_account_name, container_name)


In [0]:
# adls_path
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/order_items/landing"

# checkpoint folder for streaming(bronze, silver, gold)
bronze_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/bronze/fact_order_items"

In [0]:
# Autoloader ingest: order_items CSV → bronze table
spark.readStream \
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv")  \
 .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
 .option("cloudFiles.schemaEvolutionMode", "rescue") \
 .option("header", "true") \
 .option("cloudFiles.inferColumnTypes", "true") \
 .option("rescuedDataColumn", "_rescued_data") \
 .option("cloudFiles.includeExistingFiles", "true")  \
 .option("pathGlobFilter", "*.csv") \
 .load(adls_path) \
 .withColumn("ingest_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.col("_metadata.file_path")) \
 .writeStream \
 .outputMode("append") \
 .option("checkpointLocation", bronze_checkpoint_path) \
 .trigger(availableNow=True) \
 .toTable(f"{catalog}.bronze.order_items") \
 .awaitTermination()

In [0]:
%sql
select 
* 
from shopvista.bronze.order_items 
limit 5

## ORDER RETURNS

In [0]:
# adls_path
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/order_returns/landing"

# checkpoint folder for streaming(bronze, silver, gold)
bronze_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/bronze/fact_order_returns"

In [0]:
# Autoloader ingest: order_returns CSV → bronze table
spark.readStream \
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv")  \
 .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
 .option("cloudFiles.schemaEvolutionMode", "rescue") \
 .option("header", "true") \
 .option("cloudFiles.inferColumnTypes", "true") \
 .option("rescuedDataColumn", "_rescued_data") \
 .option("cloudFiles.includeExistingFiles", "true")  \
 .option("pathGlobFilter", "*.csv") \
 .load(adls_path) \
 .withColumn("ingest_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.col("_metadata.file_path")) \
 .writeStream \
 .outputMode("append") \
 .option("checkpointLocation", bronze_checkpoint_path) \
 .trigger(availableNow=True) \
 .toTable(f"{catalog}.bronze.order_returns") \
 .awaitTermination()

In [0]:
%sql
select 
* 
from shopvista.bronze.order_returns
limit 5

ORDER SHIPMENTS

In [0]:
# adls_path
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/order_shipments/landing"

# checkpoint folder for streaming(bronze, silver, gold)
bronze_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/bronze/fact_order_shipments"

In [0]:
# Autoloader ingest: order_shipments CSV → bronze table
spark.readStream \
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv")  \
 .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
 .option("cloudFiles.schemaEvolutionMode", "rescue") \
 .option("header", "true") \
 .option("cloudFiles.inferColumnTypes", "true") \
 .option("rescuedDataColumn", "_rescued_data") \
 .option("cloudFiles.includeExistingFiles", "true")  \
 .option("pathGlobFilter", "*.csv") \
 .load(adls_path) \
 .withColumn("ingest_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.col("_metadata.file_path")) \
 .writeStream \
 .outputMode("append") \
 .option("checkpointLocation", bronze_checkpoint_path) \
 .trigger(availableNow=True) \
 .toTable(f"{catalog}.bronze.order_shipments") \
 .awaitTermination()

In [0]:
%sql
select 
* 
from shopvista.bronze.order_shipments
limit 5